In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.3 SwiGLU

The classic MLP is `Linear -> GELU -> Linear`. SwiGLU adds a multiplicative gate:
`down( silu(gate(x)) * up(x) )`. The gate lets the network modulate information
multiplicatively, which is more expressive per parameter.

In [ ]:
from pocketlm import SwiGLU
from torch import nn

dim = 16
swiglu = SwiGLU(dim)
gelu_mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
x = torch.randn(2, 5, dim)
print("SwiGLU preserves shape:", tuple(swiglu(x).shape))
print("SwiGLU params:", sum(p.numel() for p in swiglu.parameters()),
      " GELU-MLP params:", sum(p.numel() for p in gelu_mlp.parameters()))

SwiGLU uses three projections (gate, up, down) instead of two; implementations
usually shrink the hidden size to keep the budget equal. The gain is the gating,
not the parameter count.

In [ ]:
assert tuple(swiglu(x).shape) == (2, 5, dim)